In [8]:
import pandas as pd
import numpy as np
import os   
import time
from autogluon.tabular import TabularPredictor

def split_datasets_random(datasets, train_ratio=0.8, seed=42):
    train_sets = {}
    test_sets = {}
    
    # This loop keeps dataset structure intact
    for name, df in datasets.items():
        if "Dataset" not in df.columns:
            raise ValueError(f"Dataset '{name}' is missing 'Dataset' column")
        
        # Get all unique dataset names in this dataframe
        all_names = df["Dataset"].unique().tolist()
        
        # Shuffle identically to split_h5()
        np.random.seed(seed)
        np.random.shuffle(all_names)
        
        # Split dataset names into train/test sets
        n_train = int(len(all_names) * train_ratio)
        train_names = set(all_names[:n_train])
        test_names = set(all_names[n_train:])
        
        # Assign rows based on dataset name
        train_df = df[df["Dataset"].isin(train_names)].reset_index(drop=True)
        test_df = df[df["Dataset"].isin(test_names)].reset_index(drop=True)
        
        train_sets[name] = train_df
        test_sets[name] = test_df
    
    return train_sets, test_sets

def prepare_for_autogluon(df):
    """
    Prepare dataframe for AutoGluon by dropping only the 'Dataset' column.

    Parameters:
        df (pd.DataFrame): input dataframe

    Returns:
        pd.DataFrame ready for AutoGluon (features + target only)
    """
    return df.drop(columns=["Dataset"], errors='ignore')

def autogluon_automation(train_df, test_df, dataset_name, target, 
                         base_path="Autogluon Models",
                         presets="medium_quality", 
                         metric="r2",
                         num_bag_folds=5, 
                         num_stack_levels=1, 
                         verbosity=1):
    
    start_time = time.time()  # Start timer
    
    # Drop other target columns if they exist
    other_targets = [col for col in ["CO_tbu", "MP_CF3"] if col != target]
    train_df = train_df.drop(columns=other_targets, errors="ignore")
    test_df = test_df.drop(columns=other_targets, errors="ignore")

    assert target in train_df.columns, f"{target} missing from train_df!"
    assert target in test_df.columns, f"{target} missing from test_df!"
    for col in other_targets:
        assert col not in train_df.columns, f"{col} still in train_df!"
        assert col not in test_df.columns, f"{col} still in test_df!"

    # Create folder hierarchy with target as top-level
    model_name = f"{target}_{dataset_name}_{presets}"
    model_path = os.path.join(base_path, target, dataset_name, presets, model_name)

    # Load the model if it already exists (so results are still reported); otherwise train it
    model_loaded = os.path.exists(model_path) and os.listdir(model_path)
    if model_loaded:
        print(f"Model at '{model_path}' exists. Loading it instead of retraining.")
        predictor = TabularPredictor.load(model_path)
    else:
        os.makedirs(model_path, exist_ok=True)

        predictor = TabularPredictor(label=target, path=model_path, eval_metric=metric).fit(
            train_data=train_df,
            presets=presets,
            num_bag_folds=num_bag_folds,
            num_stack_levels=num_stack_levels,
            verbosity=verbosity
        )

    # Get CV R² explicitly regardless of optimization metric
    lb = predictor.leaderboard(silent=True)

    if 'score_val' in lb.columns:
        # Sort leaderboard by score_val (descending)
        lb_sorted = lb.sort_values(by='score_val', ascending=False)
        cv_r2 = lb_sorted.iloc[0]['score_val']
    else:
        cv_r2 = None

    # Evaluate on test set
    test_results = predictor.evaluate(test_df, silent=True)

    # Extract R2 and RMSE helper
    def extract_metrics(results):
        r2 = results.get('r2', None)
        rmse = results.get('root_mean_squared_error', None)
        if rmse is None:
            rmse = results.get('rmse', None)
        return r2, rmse

    test_r2, test_rmse = extract_metrics(test_results)

    elapsed_time = None if model_loaded else time.time() - start_time  # Training time (None for loaded models)

    # Flip sign of RMSE
    test_rmse_flipped = -test_rmse if test_rmse is not None else None

    results = {
        'Dataset': dataset_name, 'Preset': presets, 'Metric': metric, 'Target': target,
        'CV_R2': cv_r2,
        'Test_R2': test_r2, 'Test_RMSE': test_rmse_flipped,
        'Elapsed_Time_sec': elapsed_time
    }

    print(f"Results for {dataset_name} | Target: {target} | Preset: {presets} | Metric: {metric}:")
    print(f"  CV R2:    {cv_r2:.3f}" if cv_r2 is not None else "  CV R2:    N/A")
    print(f"  Test R2:  {test_r2:.3f}, Test RMSE:  {test_rmse_flipped:.3f}")
    print(f"  Elapsed Time: {elapsed_time:.2f} seconds\n" if elapsed_time is not None else "  Elapsed Time: N/A (loaded existing model)\n")

    return results

def batch_train_autogluon(train_sets, test_sets, targets, presets_list,
                          base_path="AutogluonModels",
                          num_bag_folds=5, num_stack_levels=1, verbosity=2):
    results_list = []

    for dataset_name in train_sets.keys():
        train_df = train_sets[dataset_name]
        test_df = test_sets[dataset_name]

        for target in targets:
            for preset in presets_list:

                print(f"\n=== Training: Dataset={dataset_name}, Target={target}, Preset={preset} ===")
                
                res = autogluon_automation(
                    train_df=train_df,
                    test_df=test_df,
                    dataset_name=dataset_name,
                    target=target,
                    base_path=base_path,
                    presets=preset,
                    metric='r2',
                    num_bag_folds=num_bag_folds,
                    num_stack_levels=num_stack_levels,
                    verbosity=verbosity
                )
                
                # Append result only if training happened (res is not None)
                if res is not None:
                    results_list.append(res)

    # Convert results list of dicts into a DataFrame
    return pd.DataFrame(results_list)

def split_by_dataset_tag(csv_path, tag):
    # Load data
    df = pd.read_csv(csv_path)

    # Create boolean mask
    test_mask = df["Dataset"].astype(str).str.contains(tag, na=False)

    # Split
    test_df = df[test_mask]
    train_df = df[~test_mask]

    print(f"Test set shape ({tag}): {test_df.shape}")
    print(f"Train set shape ({tag}): {train_df.shape}")

    # Output folder
    out_dir = "Featurized datasets/Training Datasets/Leave One Ligand Out Datasets"

    # Ensure folder exists
    os.makedirs(out_dir, exist_ok=True)

    # Save paths
    test_path = os.path.join(out_dir, f"Dataset 4 - {tag} test.csv")
    train_path = os.path.join(out_dir, f"Dataset 4 - {tag} train.csv")

    # Save
    test_df.to_csv(test_path, index=False)
    train_df.to_csv(train_path, index=False)

    return 

def load_train_test_sets(ligand_ids, base_dir):
    train_sets = {}
    test_sets = {}

    for lig in ligand_ids:
        train_path = os.path.join(base_dir, f"Dataset 4 - {lig} train.csv")
        test_path  = os.path.join(base_dir, f"Dataset 4 - {lig} test.csv")

        # Load directly into memory
        train_sets[lig] = pd.read_csv(train_path)
        test_sets[lig]  = pd.read_csv(test_path)

    return train_sets, test_sets

In [9]:
dataset = "Featurized datasets/Training Datasets/Dataset 4.csv"
ligand_ids = ["L02", "L05", "L04", "L33", "L32", "L48", "L56", "L12"]
base_dir = "Featurized datasets/Training Datasets/Leave One Ligand Out Datasets"


for tag in ligand_ids:
    split_by_dataset_tag(dataset, tag)

train_sets, test_sets = load_train_test_sets(ligand_ids, base_dir)

Test set shape (L02): (20, 71)
Train set shape (L02): (180, 71)
Test set shape (L03): (20, 71)
Train set shape (L03): (180, 71)
Test set shape (L12): (20, 71)
Train set shape (L12): (180, 71)
Test set shape (L13): (20, 71)
Train set shape (L13): (180, 71)
Test set shape (L25): (20, 71)
Train set shape (L25): (180, 71)
Test set shape (L26): (20, 71)
Train set shape (L26): (180, 71)
Test set shape (L34): (20, 71)
Train set shape (L34): (180, 71)
Test set shape (L35): (20, 71)
Train set shape (L35): (180, 71)


In [ ]:
results = batch_train_autogluon(
    train_sets,
    test_sets,
    base_path= "Autogluon Models/Leave One Ligand Out",
    targets=["CO_tbu", "MP_CF3"],
    presets_list=["medium_quality"]
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.8
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Wed Nov  5 21:30:44 PST 2025; root:xnu-11417.140.69.705.2~1/RELEASE_ARM64_T6041
CPU Count:          14
Memory Avail:       9.87 GB / 24.00 GB (41.1%)
Disk Space Avail:   623.54 GB / 926.35 GB (67.3%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ...
AutoGluon will save models to "/Users/andytran/Documents/Nickel Library/Nickel Spectral Fingerprints backup/Autogluon Models/Leave One Ligand Out/CO_tbu/L02/medium_quality/CO_tbu_L02_medium_quality"
Train Data Rows:    180
Train Data Columns: 69
Label Column:       CO_tbu
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (1.0287854159379437, 0.0, 0.31019, 0.311


=== Training: Dataset=L02, Target=CO_tbu, Preset=medium_quality ===


	Useless Original Features (Count: 2): ['Metal_Lambda_4', 'Metal_Lambda_5']
		These features carry no predictive signal and should be manually investigated.
		This is typically a feature which has the same value for all rows.
		These features do not need to be present at inference time.
	Unused Original Features (Count: 11): ['Dataset', 'Delta_Sample_Lambda_Metal_Lambda_1_4', 'Delta_Sample_Lambda_Metal_Lambda_1_5', 'Delta_Sample_Lambda_Metal_Lambda_2_4', 'Delta_Sample_Lambda_Metal_Lambda_2_5', 'Delta_Sample_Lambda_Metal_Lambda_3_4', 'Delta_Sample_Lambda_Metal_Lambda_3_5', 'Delta_Sample_Lambda_Metal_Lambda_4_4', 'Delta_Sample_Lambda_Metal_Lambda_4_5', 'Delta_Sample_Lambda_Metal_Lambda_5_4', 'Delta_Sample_Lambda_Metal_Lambda_5_5']
		These features were not used to generate any of the output features. Add a feature generator compatible with these features to utilize them.
		Features can also be unused if they carry very little information, such as being categorical but having almost entir

Results for L02 | Target: CO_tbu | Preset: medium_quality | Metric: r2:
  CV R2:    0.571
  Test R2:  0.519, Test RMSE:  0.209
  Elapsed Time: 74.15 seconds


=== Training: Dataset=L02, Target=MP_CF3, Preset=medium_quality ===


	0.4973	 = Validation score   (r2)
	0.73s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ...
	Fitting 5 child models (S1F1 - S1F5) | Fitting with ParallelLocalFoldFittingStrategy (5 workers, per: cpus=2, gpus=0, memory=0.13%)
	0.4945	 = Validation score   (r2)
	0.64s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ...
/Users/andytran/.pyenv/versions/nickelLibrary/lib/python3.11/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
	0.4996	 = Validation score   (r2)
	0.16s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ...
	Fitting 5 child models (S1F1 - S1F5) | Fitting with ParallelLocalFoldFittingStrategy (5 workers, per: cpus=2, gpus=0, memory=4.18%)
	0.4831

...

Results for L34 | Target: MP_CF3 | Preset: medium_quality | Metric: r2:
  CV R2:    0.571
  Test R2:  0.611, Test RMSE:  0.127
  Elapsed Time: 42.37 seconds


=== Training: Dataset=L35, Target=CO_tbu, Preset=medium_quality ===


	0.5815	 = Validation score   (r2)
	2.06s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ...
	Fitting 5 child models (S1F1 - S1F5) | Fitting with ParallelLocalFoldFittingStrategy (5 workers, per: cpus=2, gpus=0, memory=0.12%)
	0.5489	 = Validation score   (r2)
	1.17s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ...
/Users/andytran/.pyenv/versions/nickelLibrary/lib/python3.11/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
	0.5043	 = Validation score   (r2)
	0.15s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ...
	Fitting 5 child models (S1F1 - S1F5) | Fitting with ParallelLocalFoldFittingStrategy (5 workers, per: cpus=2, gpus=0, memory=4.07%)
	0.5463

Results for L35 | Target: CO_tbu | Preset: medium_quality | Metric: r2:
  CV R2:    0.623
  Test R2:  -15.047, Test RMSE:  0.350
  Elapsed Time: 38.28 seconds


=== Training: Dataset=L35, Target=MP_CF3, Preset=medium_quality ===


	0.5546	 = Validation score   (r2)
	0.66s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ...
	Fitting 5 child models (S1F1 - S1F5) | Fitting with ParallelLocalFoldFittingStrategy (5 workers, per: cpus=2, gpus=0, memory=0.12%)
	0.5546	 = Validation score   (r2)
	0.66s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ...
/Users/andytran/.pyenv/versions/nickelLibrary/lib/python3.11/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
	0.5762	 = Validation score   (r2)
	0.16s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ...
	Fitting 5 child models (S1F1 - S1F5) | Fitting with ParallelLocalFoldFittingStrategy (5 workers, per: cpus=2, gpus=0, memory=4.20%)
	0.5701

Results for L35 | Target: MP_CF3 | Preset: medium_quality | Metric: r2:
  CV R2:    0.592
  Test R2:  -4.478, Test RMSE:  0.193
  Elapsed Time: 29.83 seconds



In [14]:
print(results)
results.to_excel("Autogluon Results/Leave One Ligand Out - medium_quality.xlsx", index=False)


   Dataset          Preset Metric  Target     CV_R2    Test_R2  Test_RMSE  \
0      L02  medium_quality     r2  CO_tbu  0.571336   0.518575   0.209295   
1      L02  medium_quality     r2  MP_CF3  0.526791   0.747823   0.115984   
2      L03  medium_quality     r2  CO_tbu  0.567937   0.038848   0.321306   
3      L03  medium_quality     r2  MP_CF3  0.569329   0.578462   0.108866   
4      L12  medium_quality     r2  CO_tbu  0.623004  -7.336738   0.231167   
5      L12  medium_quality     r2  MP_CF3  0.562445 -53.371047   0.290049   
6      L13  medium_quality     r2  CO_tbu  0.585613   0.000000   0.352666   
7      L13  medium_quality     r2  MP_CF3  0.577594  -4.386124   0.186480   
8      L25  medium_quality     r2  CO_tbu  0.683558  -0.012582   0.307230   
9      L25  medium_quality     r2  MP_CF3  0.528679   0.395645   0.185509   
10     L26  medium_quality     r2  CO_tbu  0.638412   0.457648   0.209323   
11     L26  medium_quality     r2  MP_CF3  0.566436   0.487258   0.126762   

In [15]:
results_best_quality = batch_train_autogluon(
    train_sets,
    test_sets,
    base_path= "Autogluon Models/Leave One Ligand Out",
    targets=["CO_tbu", "MP_CF3"],
    presets_list=["best_quality"]
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.8
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Wed Nov  5 21:30:44 PST 2025; root:xnu-11417.140.69.705.2~1/RELEASE_ARM64_T6041
CPU Count:          14
Memory Avail:       9.80 GB / 24.00 GB (40.8%)
Disk Space Avail:   622.59 GB / 926.35 GB (67.2%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the 


=== Training: Dataset=L02, Target=CO_tbu, Preset=medium_quality ===
Model at 'Autogluon Models/Leave One Ligand Out/CO_tbu/L02/medium_quality/CO_tbu_L02_medium_quality' exists. Skipping training.

=== Training: Dataset=L02, Target=CO_tbu, Preset=best_quality ===


Leaderboard on holdout data (DyStack):
                           model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0                 XGBoost_BAG_L2       0.624779   0.528496          r2        0.261655       0.121250   6.272389                 0.016705                0.005099           0.377085            2       True        116
1             XGBoost_r33_BAG_L2       0.598234   0.536737          r2        0.271387       0.122933   7.073347                 0.026437                0.006782           1.178043            2       True        126
2             XGBoost_r95_BAG_L2       0.594243   0.496150          r2        0.258310       0.121405   6.391876                 0.013360                0.005253           0.496572            2       True        211
3             XGBoost_r49_BAG_L2       0.589048   0.545680          r2        0.259370       0.12

Results for L02 | Target: CO_tbu | Preset: best_quality | Metric: r2:
  CV R2:    0.588
  Test R2:  0.504, Test RMSE:  0.212
  Elapsed Time: 1363.31 seconds


=== Training: Dataset=L02, Target=MP_CF3, Preset=medium_quality ===
Model at 'Autogluon Models/Leave One Ligand Out/MP_CF3/L02/medium_quality/MP_CF3_L02_medium_quality' exists. Skipping training.

=== Training: Dataset=L02, Target=MP_CF3, Preset=best_quality ===


Leaderboard on holdout data (DyStack):
                           model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        RandomForest_r15_BAG_L1       0.595791   0.496725          r2        0.029069       0.029673   0.193821                 0.029069                0.029673           0.193821            1       True         79
1        RandomForest_r16_BAG_L1       0.587621   0.491666          r2        0.032998       0.032170   0.247165                 0.032998                0.032170           0.247165            1       True         92
2         RandomForestMSE_BAG_L1       0.587621   0.491666          r2        0.033185       0.031855   0.299566                 0.033185                0.031855           0.299566            1       True          5
3       RandomForest_r195_BAG_L1       0.583624   0.494369          r2        0.031276       0.03

...

Results for L34 | Target: MP_CF3 | Preset: best_quality | Metric: r2:
  CV R2:    0.605
  Test R2:  0.653, Test RMSE:  0.120
  Elapsed Time: 1025.48 seconds


=== Training: Dataset=L35, Target=CO_tbu, Preset=medium_quality ===
Model at 'Autogluon Models/Leave One Ligand Out/CO_tbu/L35/medium_quality/CO_tbu_L35_medium_quality' exists. Skipping training.

=== Training: Dataset=L35, Target=CO_tbu, Preset=best_quality ===


Leaderboard on holdout data (DyStack):
                           model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0            XGBoost_r194_BAG_L1       0.603965   0.455828          r2        0.023571       0.009527   0.437524                 0.023571                0.009527           0.437524            1       True         33
1             XGBoost_r98_BAG_L1       0.578486   0.473987          r2        0.039705       0.013228   1.517813                 0.039705                0.013228           1.517813            1       True         47
2       RandomForest_r127_BAG_L2       0.576512   0.511360          r2        0.172942       0.183067   7.608346                 0.017984                0.030058           0.211934            2       True        163
3        RandomForest_r15_BAG_L2       0.575861   0.519599          r2        0.173446       0.18

Results for L35 | Target: CO_tbu | Preset: best_quality | Metric: r2:
  CV R2:    0.691
  Test R2:  -14.372, Test RMSE:  0.343
  Elapsed Time: 1315.81 seconds


=== Training: Dataset=L35, Target=MP_CF3, Preset=medium_quality ===
Model at 'Autogluon Models/Leave One Ligand Out/MP_CF3/L35/medium_quality/MP_CF3_L35_medium_quality' exists. Skipping training.

=== Training: Dataset=L35, Target=MP_CF3, Preset=best_quality ===


Leaderboard on holdout data (DyStack):
                           model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        RandomForest_r16_BAG_L2       0.843061   0.450004          r2        0.168612       0.153707  18.547934                 0.020454                0.032229           0.268857            2       True        199
1         RandomForestMSE_BAG_L2       0.843061   0.450004          r2        0.178144       0.153500  18.447889                 0.029987                0.032022           0.168813            2       True        112
2       RandomForest_r195_BAG_L2       0.840059   0.454423          r2        0.178824       0.152389  18.482364                 0.030667                0.030911           0.203288            2       True        131
3        RandomForest_r39_BAG_L2       0.823634   0.451966          r2        0.180248       0.15

Results for L35 | Target: MP_CF3 | Preset: best_quality | Metric: r2:
  CV R2:    0.648
  Test R2:  -4.761, Test RMSE:  0.198
  Elapsed Time: 996.36 seconds



In [16]:
results_best_quality.to_excel("Autogluon Results/Leave One Ligand Out - best_quality.xlsx", index=False)
